In [17]:
from langgraph.graph import StateGraph,END,START
from langgraph.types import interrupt,Command
from langgraph.checkpoint.memory import InMemorySaver
from langchain_google_genai import ChatGoogleGenerativeAI
from typing  import TypedDict,Annotated
from langgraph.graph.message import add_messages
from langchain_core.messages import BaseMessage,AIMessage

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [16]:
class ChatState(TypedDict):
    message:Annotated[list[BaseMessage],add_messages]

In [3]:
class AgentState(TypedDict):
    action:str
    approved: bool|None 
    result: str

In [5]:
def agent_node(state:AgentState):
    return {"action":"delete_files('/data/old_logs')"}

def review_node(state:AgentState):
    human_decison=interrupt({
        "question":"Approve the action?",
        "action":state['action']
    })
    return {"approved": human_decison=='yes'}

def execute_node(state:AgentState):
    if state['approved']:
        return {"result":f"Executed {state['action']}"}
    return {"result":"Action not approved"}

In [6]:
builder=StateGraph(AgentState)
builder.add_node('agent',agent_node)
builder.add_node('review',review_node)
builder.add_node('execute',execute_node)

builder.add_edge(START,'agent')
builder.add_edge('agent','review')
builder.add_edge('review','execute')
builder.add_edge('execute',END)

checkpointer=InMemorySaver()
graph=builder.compile(checkpointer)

In [7]:
config={'configurable':{'thread_id':'1234'}}

for event in graph.stream({"action":None,'approved':None,'result':None},config):
    print(event)

{'agent': {'action': "delete_files('/data/old_logs')"}}
{'__interrupt__': (Interrupt(value={'question': 'Approve the action?', 'action': "delete_files('/data/old_logs')"}, id='26c694cedd92a39c5de411368364ee45'),)}


In [8]:
for event in graph.stream(Command(resume="yes"),config):
    print(event)

{'review': {'approved': True}}
{'execute': {'result': "Executed delete_files('/data/old_logs')"}}


In [10]:
model=ChatGoogleGenerativeAI(model='gemini-2.5-flash')

In [19]:
def chat_node(state:ChatState):
    decision=interrupt({
        "type":"approval",
        "reason":"Model is about to answer user question",
        "question":state["message"][-1].content,
        "instuctions":"Approve this question yes/no"
    })
    if decision["approved"]=="no":
        return {"message":AIMessage(content="Not approved")}
    else:
        response=model.invoke(state["message"])
        return {"message":response}

In [20]:
graph=StateGraph(ChatState)
graph.add_node('chat',chat_node)
graph.add_edge(START,'chat')
graph.add_edge('chat',END)

In [21]:
workflow=graph.compile(InMemorySaver())

In [22]:
config={'configurable':{'thread_id':'5678'}}

intial_input={
    "message":[
        ("user","Explain gradient descent in simple terms")
    ]
}

result=workflow.invoke(intial_input,config)

In [23]:
result

{'message': [HumanMessage(content='Explain gradient descent in simple terms', additional_kwargs={}, response_metadata={}, id='cde6f766-01e5-4ea7-8827-d95783d88928')],
 '__interrupt__': [Interrupt(value={'type': 'approval', 'reason': 'Model is about to answer user question', 'question': 'Explain gradient descent in simple terms', 'instuctions': 'Approve this question yes/no'}, id='82f3c4813195707bf6d0f67fa7f26879')]}

In [24]:
user_input=input("approve this question yes/no")

In [25]:
final_result=workflow.invoke(
    Command(resume={'approved':user_input}),
    config=config
)

In [28]:
final_result

{'message': [HumanMessage(content='Explain gradient descent in simple terms', additional_kwargs={}, response_metadata={}, id='cde6f766-01e5-4ea7-8827-d95783d88928'),
  AIMessage(content='Imagine you\'re blindfolded and standing somewhere on a mountain range. Your goal is to find the lowest point in the entire landscape (the bottom of a valley).\n\nYou can\'t see the whole mountain, but you can feel the ground directly beneath your feet.\n\nHere\'s how you\'d try to find the lowest point:\n\n1.  **Feel the Slope:** You\'d feel around with your feet to figure out which direction is *most downhill* right where you\'re standing.\n2.  **Take a Step:** You\'d take a small step in that most downhill direction.\n3.  **Repeat:** You\'d repeat this process: feel the slope, take a step downhill, feel the slope, take a step downhill...\n\nIf you keep doing this, step by step, you\'ll eventually reach the bottom of the valley.\n\n---\n\n**Now, let\'s translate this to Gradient Descent in Machine Le